# 02 — Vietnamese adaptation of the actual pretrained TITL 8M

Use the WAV/text corpus from [01 — T4 data](https://colab.research.google.com/github/Huyvux12/Ancaptritue/blob/main/notebooks/01_MaiChi_Data_T4.ipynb). Existing COMPLETE v3 bundles can be reused: **do not regenerate WAVs just to change the student**.

This notebook loads the released **af_heart** checkpoint and the original Paradee/TITL `s+mlp` text model + `A` waveform decoder. Vietnamese tone-bearing character embeddings are appended; all original weights are retained. Complete inference student: **8,110,486 parameters** including waveform generation. No external neural vocoder.

**Experimental Vietnamese adaptation, not a released Vietnamese checkpoint.** ZeroTTS WAV supervision differs from the paper's Kokoro internal-feature/shared-excitation recipe. Loudness is an approximate surrogate for the learned `N` signal. English baseline parity is tested; Vietnamese quality is not guaranteed. Small-set listening/independent ASR must pass before full training/export. See [design and failure diagnosis](https://github.com/Huyvux12/Ancaptritue/blob/main/docs/TITL8M_VI_DESIGN.md).

## 1. Runtime and paths

Use the same `MODE` / `DRIVE_PARENT` as notebook 01. Start with pilot to test execution; **full training needs the full corpus**. On an external RTX runtime set `USE_DRIVE=False` and provide an already mounted/synced directory. Train on SSD; snapshots go to a separate TITL experiment directory. Defaults are starting budgets, not measured convergence or timing estimates.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, hashlib
os.environ['ORT_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['HF_HUB_DISABLE_XET'] = '1'
USE_DRIVE = True
MODE = 'pilot'                    # 'pilot' or 'full', same as notebook 01
RUN_LABEL = 'titl8m_vi_01'         # New label AND local root for a new experiment
DRIVE_PARENT = Path('/content/drive/MyDrive/MaiChi_Distillation_v03')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
assert MODE in ['pilot', 'full']
BUNDLE = DRIVE_PARENT / 'datasets' / MODE
BACKUP = DRIVE_PARENT / 'training_titl8m' / MODE / RUN_LABEL
ROOT = Path('/content/maichi_titl8m') / MODE / RUN_LABEL
RAW_ROOT = Path('/content/maichi_raw_titl8m') / MODE / RUN_LABEL
REPO_DIR = Path('/content/Ancaptritue_titl8m')
BATCH_SIZE = 8
WORKERS = 2                      # Set 0 if multiprocessing is unavailable
OVERFIT_ITEMS = 32
SMALL_BRIDGE_STEPS = 2000
SMALL_JOINT_STEPS = 4000
FULL_BRIDGE_STEPS = 8000
FULL_JOINT_STEPS = 20000
GAN_STEPS = 5000
RUN_FULL_ADAPTATION = (MODE == 'full')
RUN_GAN = False                  # Optional; only after a validation ASR gate
RUN_BENCHMARK = False
RUN_OFFICIAL_SCORE = False       # Extra evaluator models/dependencies
assert (BUNDLE / 'bundle.json').exists(), 'Finish notebook 01: COMPLETE bundle required'
ROOT.mkdir(parents=True, exist_ok=True)
print('Local SSD:', shutil.disk_usage(ROOT))
print('Data:', BUNDLE, '\nCheckpoints:', BACKUP)

## 2. Clone training source and pin resume

The raw-data producer commit is provenance for the data only; old producer commits have no TITL adapter. A fresh experiment uses current `main`. A Drive resume automatically checks out the **training** commit recorded in `latest.json`. Source hashes must match before checkpoint resume. Use a fresh experiment directory when changing source or data.

In [ ]:
REPO_URL = 'https://github.com/Huyvux12/Ancaptritue.git'
saved_backup = json.loads((BACKUP / 'latest.json').read_text()) if (BACKUP / 'latest.json').exists() else {}
REPO_REF = saved_backup.get('project_commit') or 'main'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
assert not subprocess.check_output(['git', '-C', str(REPO_DIR), 'status', '--porcelain'], text=True).strip(), 'Protect local source edits; choose another REPO_DIR'
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', 'origin/main' if REPO_REF == 'main' else REPO_REF], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert (REPO_DIR / 'titl_vi/cli.py').exists(), 'This commit predates the TITL adapter'
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Training commit:', COMMIT)

## 3. Dependencies and real CUDA check

Keep the runtime's CUDA PyTorch. **FP32** training avoids introducing an untested LSTM/iSTFT mixed-precision path. The training workflow does not load ZeroTTS teacher weights; `zerotts` is used only for pinned Vietnamese number normalization. Restart a previously used kernel if an already imported package version was replaced.

In [ ]:
assert (3, 11) <= sys.version_info[:2] <= (3, 12), 'Use Python 3.11–3.12'
def pip_install(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *map(str, args)], check=True)
pip_install('-r', REPO_DIR / 'requirements-titl-vi.txt')
pip_install('pyworld==0.3.5', '--no-build-isolation')
pip_install('zerotts==0.1.5', '--no-deps')
import torch
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 6), 'Use a CUDA Torch >=2.6 runtime for safe pretrained CTC loading'
assert torch.cuda.is_available(), 'Connect the intended CUDA runtime first'
x = torch.randn(128, 128, device='cuda')
assert torch.isfinite(x @ x).all()
torch.cuda.synchronize()
print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory / 2**30, 'GiB')
(ROOT / 'pip_freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

## 4. Restore checkpoints, import raw corpus, verify pretrained English

Only WAV/text/splits and benchmark references are imported. **Old 256-hop features are skipped.** This adapter recomputes 40-Hz character timing and 80-Hz F0/loudness for the original decoder. The restored checkpoint/optimizer lives on SSD. Rerunning preparation rebinds raw WAV paths after VM loss.

English samples check the weight-loading path. Listen for the pretrained baseline before spending GPU time on Vietnamese.

In [ ]:
from titl_vi.storage import restore, backup
from titl_vi.data import restore_raw, atomic_json
from titl_vi.train import source_hashes
restore(BACKUP, ROOT)
identity = {'commit': COMMIT, 'package_sha256': source_hashes()}
marker = ROOT / 'source_revision.json'
if marker.exists():
    assert json.loads(marker.read_text()) == identity, 'Source changed: use a new ROOT/RUN_LABEL'
else:
    atomic_json(marker, identity)
def run(stage, **kwargs):
    cmd = [sys.executable, '-u', '-m', 'titl_vi.cli', stage, '--root', str(ROOT)]
    for key, value in kwargs.items():
        if value is not None: cmd.extend(['--' + key.replace('_', '-'), str(value)])
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
restore_raw(BUNDLE, RAW_ROOT)
if not (ROOT / 'initial.pt').exists():
    run('bootstrap', device='cuda')
meta = json.loads((ROOT / 'pretrained_metadata.json').read_text())
assert meta['budget']['total'] < 10_000_000
print(json.dumps(meta['budget'], indent=2))
from IPython.display import Audio, display
for path in sorted((ROOT / 'english_baseline').glob('*.wav'))[:2]: display(Audio(filename=str(path)))

## 5. Align and audit Vietnamese WAVs

Pinned Vietnamese CTC weights are loaded with their actual custom head, without remote-code execution. The aligner rejects poor transcript matches. WORLD extracts pitch. Train-only log-RMS statistics are moment-matched to the pretrained learned `N` range; this is approximate supervision and a key adaptation risk.

Inspect rejections, tone coverage and actual accepted counts. `spoken_text` in a raw corpus row can override automatic number expansion for a reviewed transcript. Never edit a corpus under an existing run.

In [ ]:
run('prepare', raw_root=RAW_ROOT, device='cuda')
summary = json.loads((ROOT / 'preparation_summary.json').read_text())
print(json.dumps(summary, indent=2))
from titl_vi.data import read_rows
rejected = read_rows(ROOT / 'rejected.jsonl')
print('First rejections:', rejected[:10])
assert summary['splits']['train'] >= OVERFIT_ITEMS
assert summary['splits']['val'] >= 2
backup(ROOT, BACKUP)

## 6. Small-set bridge, then joint adaptation

Use a fixed subset of 32 training utterances. Bridge keeps the pretrained decoder frozen; joint fine-tunes it with a lower learning rate while moving from measured to predicted prosody. The harmonic source remains frozen. Waveform phase differs across teachers, so use spectral losses rather than a sample-wise waveform target.

Check `runs/*/train.jsonl` and `validation.jsonl`. `best.pt` is selected by aligned validation loss, **not** by free-running speech quality. If the ASR gate below fails, stop and inspect the three diagnostic views before increasing budgets.

In [ ]:
def train_stage(name, phase, steps, init=None, overfit=0):
    run('train', run=name, phase=phase, steps=steps, init=init,
        batch_size=BATCH_SIZE, workers=WORKERS, overfit=overfit,
        device='cuda', backup_dir=BACKUP)
    path = ROOT / 'runs' / name / 'best.pt'
    assert path.exists()
    return path
small_bridge = train_stage('small_bridge', 'bridge', SMALL_BRIDGE_STEPS, overfit=OVERFIT_ITEMS)
small_joint = train_stage('small_joint', 'joint', SMALL_JOINT_STEPS, init=small_bridge, overfit=OVERFIT_ITEMS)

## 7. Listen and apply an independent intelligibility gate

Three views isolate problems: (1) aligned durations + measured F0/loudness, (2) aligned durations + predicted prosody, (3) free text only. View 1 is not an oracle reconstruction: neither CTC nor log-RMS supplies Kokoro's learned internal features.

PhoWhisper transcribes the same teacher and student utterances. Required character error rate: student ≤20% and ≤teacher+10 percentage points; teacher ≤20%; basic signal checks must pass. This is a selected-sample intelligibility check, **not proof of naturalness, speaker fidelity or six-tone accuracy**. Listen to all views and the teacher WAVs. A failed cell blocks subsequent full training.

In [ ]:
run('diagnostics', checkpoint=small_joint, name='inspect_small', split='train', count=4,
    overfit=OVERFIT_ITEMS, device='cuda')
records = json.loads((ROOT / 'diagnostics/inspect_small/samples.json').read_text())['records']
for row in records:
    print(row['text'])
    display(Audio(filename=row['source_wav']))
    for kind, path in row['audio'].items():
        print(kind); display(Audio(filename=path))
run('gate', checkpoint=small_joint, name='overfit', split='train', count=8,
    overfit=OVERFIT_ITEMS, device='cuda')
backup(ROOT, BACKUP)

## 8. Full corpus adaptation, only after the small-set gate

Requires ≥1,000 accepted train items and ≥20 validation items. Use `MODE='full'` from the start for this path; changing pilot to full changes corpus fingerprints. A validation gate is required before the joint/GAN stages. Every gate is tied to the exact initialization checkpoint SHA and corpus. Failure stops this path.

GAN refinement is optional and disabled by default. Changing the stage source/config requires a new run. Checkpoints include optimizer and RNG; the data iterator restarts on resume, so exact batch replay is not promised. No RTX time estimate is claimed until your step timings are measured.

In [ ]:
if RUN_FULL_ADAPTATION:
    assert MODE == 'full'
    full_bridge = train_stage('full_bridge', 'bridge', FULL_BRIDGE_STEPS, init=small_joint)
    run('gate', checkpoint=full_bridge, name='pre_joint', split='val', count=8, device='cuda')
    full_joint = train_stage('full_joint', 'joint', FULL_JOINT_STEPS, init=full_bridge)
    selected = full_joint
    if RUN_GAN:
        run('gate', checkpoint=full_joint, name='pre_gan', split='val', count=8, device='cuda')
        selected = train_stage('full_gan', 'gan', GAN_STEPS, init=full_joint)
else:
    selected = small_joint
    print('Pilot only. Do not label this a production Vietnamese model.')

## 9. Final validation gate and deployable text-to-WAV export

Final export requires the exact selected checkpoint to pass on validation utterances, including pilot exports. The deployed model contains one text/decoder/source state only; CTC, PhoWhisper, critics, teacher and optimizer are training/evaluation tools. Full inference uses raw Vietnamese text and internal number normalization. Output is native 24 kHz.

In [ ]:
run('gate', checkpoint=selected, name='final', split='val', count=8, device='cuda')
run('export', checkpoint=selected)
DEPLOY = ROOT / 'export/student_vi_8m.pt'
examples = ['Xin chào, mình là Mai Chi.', 'Ma, má, mà, mả, mã, mạ.',
            'Năm 2026, chúng ta thử nghiệm giọng nói tiếng Việt.']
for i, text in enumerate(examples):
    path = ROOT / 'samples' / f'vi_{i}.wav'
    run('speak', checkpoint=DEPLOY, text=text, output=path, device='cuda')
    print(text); display(Audio(filename=str(path)))
backup(ROOT, BACKUP)

## 10. Held-out benchmark and listening

Optional 137-item benchmark uses the fixed manifest reserved by notebook 01. Never tune on its public test items. The official scorer needs additional models/dependencies. WavLM similarity against same-text Mai Chi teacher audio is an adapted single-speaker measure, separate from original reference-speaker similarity. Teacher T4 vs student RTX RTF are different-hardware measurements. ASR/MOS predictors cannot replace a blinded listening study.

In [ ]:
if RUN_BENCHMARK:
    run('benchmark', raw_root=RAW_ROOT, checkpoint=DEPLOY, device='cuda')
if RUN_OFFICIAL_SCORE:
    assert RUN_BENCHMARK
    pip_install('onnxruntime==1.22.1', 'sounddevice')
    import onnxruntime as ort
    ort.disable_telemetry_events()
    import maichi_pipeline as mp
    mp.prepare_benchmark(ROOT)
    # Install the evaluator's declared requirements; keep runtime CUDA Torch.
    req = ROOT / 'ZeroBench-TTS/zerobench_eval/requirements.txt'
    if req.exists(): pip_install('-r', req)
    mp.official_score(ROOT, 'titl_vi_8m', device='cuda', skip_utmos=True)
    teacher = RAW_ROOT / 'benchmark_wavs/teacher_maichi'
    if teacher.exists():
        shutil.copytree(teacher, ROOT / 'benchmark_wavs/teacher_maichi', dirs_exist_ok=True)
        mp.paired_similarity(ROOT, 'titl_vi_8m', device='cuda')
    else:
        print('No teacher benchmark WAVs. Enable their generation in notebook 01 for paired voice comparison.')
backup(ROOT, BACKUP)
print('Export:', DEPLOY)
print('Drive recovery:', BACKUP)
print('Human listening and tone checks remain required before calling the model usable.')

## Validation scope

The repository records implementation checks in `validation/TITL8M_VI_evidence.json`. Real released weights, English load parity, original/exportable STFT comparison, Vietnamese CTC preparation, CPU gradient updates and optimizer resume were exercised. Public evidence omits corpus-derived audio metadata, hashes and ASR records.

T4/RTX throughput, full-corpus Vietnamese adaptation, tone quality and human MOS remain unverified. No quality-verified Vietnamese checkpoint is distributed. A short overfit probe does not establish generalization. If a quality gate fails in your run, inspect decoder conditioning, transcription and duration/prosody mismatch before committing to full training.